# Marching Cubes

Marching Cubes extracts a **triangular mesh** from a scalar field. It examines each cell in the grid and generates triangles where the iso-surface passes through.


In [1]:
import torch
import isoext
from isoext import viewer

# Setup: create a grid with a sphere
grid = isoext.UniformGrid([64, 64, 64])
points = grid.get_points()
grid.set_values(points.norm(dim=-1) - 0.7)


## Basic Usage

```python
vertices, faces = isoext.marching_cubes(grid, level=0.0, method="nagae")
```

- `grid`: A `UniformGrid` or `SparseGrid` with values set
- `level`: The iso-value to extract (default: 0.0)
- `method`: Algorithm variant (`"nagae"` or `"lorensen"`)


In [2]:
vertices, faces = isoext.marching_cubes(grid)

print(f"Vertices: {vertices.shape}")  # (N, 3) float32
print(f"Faces: {faces.shape}")        # (M, 3) uint32

viewer.embed(vertices, faces)


Vertices: torch.Size([9168, 3])
Faces: torch.Size([18332, 3])


## Algorithm Variants

The `method` argument selects between three variants that share the
same interface:

- `nagae` (default) produces watertight meshes.
- `lorensen` is the original 1987 algorithm; ambiguous cells can
  leave small cracks.
- `lewiner` additionally resolves topological ambiguities, following
  the field's interpolant.

See {doc}`mc_variants` for the differences in detail.


In [3]:
for method in ["nagae", "lorensen", "lewiner"]:
    v, f = isoext.marching_cubes(grid, method=method)
    print(f"{method:9s} {f.shape[0]:,} triangles")


nagae     18,332 triangles
lorensen  18,332 triangles
lewiner   18,332 triangles


## Iso-Level

The `level` parameter controls which iso-surface to extract. For signed distance fields, `level=0` gives the surface. Other values give offset surfaces:


In [4]:
# Extract at different iso-levels
for level in [-0.2, 0.0, 0.2]:
    v, f = isoext.marching_cubes(grid, level=level)
    print(f"level={level:+.1f}: {v.shape[0]:,} vertices (radius ≈ {0.7 - level:.1f})")


level=-0.2: 4,728 vertices (radius ≈ 0.9)
level=+0.0: 9,168 vertices (radius ≈ 0.7)
level=+0.2: 15,072 vertices (radius ≈ 0.5)


## Saving Meshes

Use `write_obj` to save the mesh to an OBJ file:


In [5]:
vertices, faces = isoext.marching_cubes(grid)
isoext.write_obj("mesh.obj", vertices, faces)
print("Saved to mesh.obj")


Saved to mesh.obj
